In [2]:
from pathlib import Path
import sys
import torch 

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from instancepipelineprior import get_data_loaders
from bayesianprior.discretizedtrain import EarthquakeCNN, evaluate_metrics, logits_to_magnitude

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = PROJECT_ROOT / "bayesianprior" / "data" / "best_model_huberandcross1s075.pth"

checkpoint= torch.load(checkpoint_path, map_location=device)
model = EarthquakeCNN().to(device)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()




Train samples: 979487
Validation samples: 96993
Test samples: 82769
Train batches: 7653
Val batches: 758
Test batches: 647
Train samples: 979487
Validation samples: 96993
Test samples: 82769
Train batches: 7653
Val batches: 758
Test batches: 647


EarthquakeCNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv1d(3, 16, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (3): Sequential(
      (0): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (4): Sequential(
      (0): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, str

In [3]:
train_loader,val_loader,test_loader = get_data_loaders()


In [4]:
import torch

smoothed_prior = torch.load(
    "data/smoothed_magnitude_prior.pt",
    weights_only=False
)

In [5]:
smoothed_prior = torch.tensor(
    smoothed_prior,
    dtype=torch.float32
)

In [6]:
def logits_to_magnitude_with_prior(logits,bin_centers,prior_probs,alpha):
    prior_probs = prior_probs.to(logits.device)
    bin_centers = bin_centers.to(logits.device)
    log_prior = torch.log(prior_probs.clamp(min=1e-8))

    adjusted_logits = (logits + (alpha - 1.0) * log_prior.unsqueeze(0))
    adjusted_probs = torch.softmax(adjusted_logits,dim=1)
    predictions = torch.sum(adjusted_probs* bin_centers.unsqueeze(0),dim=1)
    return predictions, adjusted_probs

In [7]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [8]:
waveforms, magnitudes = next(
    iter(val_loader)
)

waveforms = waveforms.to(device)
magnitudes = magnitudes.to(device)

print(waveforms.shape)
print(magnitudes.shape)

torch.Size([128, 3, 300])
torch.Size([128])


In [9]:
MIN_MAGNITUDE = 0.0
MAX_MAGNITUDE = 6.6
BIN_WIDTH = 0.1

BIN_CENTERS = torch.arange(
    MIN_MAGNITUDE + BIN_WIDTH / 2,
    MAX_MAGNITUDE,
    BIN_WIDTH,
    dtype=torch.float32
)

In [10]:
logits = model(waveforms)
predictions, probs = logits_to_magnitude_with_prior(logits,BIN_CENTERS,smoothed_prior,alpha=1.0)

In [11]:
original_predictions = logits_to_magnitude(
    logits,
    BIN_CENTERS.to(device)
)

print(
    torch.max(
        torch.abs(
            original_predictions - predictions
        )
    )
)

print(
    torch.allclose(
        original_predictions,
        predictions,
        atol=1e-6
    )
)

tensor(0., device='cuda:0', grad_fn=<MaxBackward1>)
True


In [12]:
import pandas as pd

In [13]:
def evaluate_metrics_with_prior(model, dataloader, device, prior_probs, alpha):
    model.eval()

    all_predictions = []
    all_targets = []
    bin_centers = BIN_CENTERS.to(device)

    with torch.no_grad():
        for waveforms, magnitudes in dataloader:
            waveforms = waveforms.to(device, non_blocking=True)
            magnitudes = magnitudes.to(device, non_blocking=True)

            logits = model(waveforms)

            predictions, _ = logits_to_magnitude_with_prior(
                logits,
                bin_centers,
                prior_probs,
                alpha
            )

            all_predictions.append(predictions.cpu())
            all_targets.append(magnitudes.cpu())

    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    errors = all_predictions - all_targets

    mae = torch.abs(errors).mean().item()
    rmse = torch.sqrt(torch.mean(errors ** 2)).item()
    bias = errors.mean().item()
    accuracy = (torch.abs(errors) <= 0.2).float().mean().item() * 100

    results = pd.DataFrame({
        "true_magnitude": all_targets.numpy(),
        "predicted_magnitude": all_predictions.numpy()
    })

    return results, mae, rmse, bias, accuracy


In [14]:
alphas = [
    0.0,
    0.25,
    0.5,
    0.75,
    1.0,
    1.25
]

In [15]:
import numpy as np

In [16]:

maelist = []
rmselist = []
biaslist = []
accuracylist = []
bin_rows = []

for alpha in alphas:
    results, mae, rmse, bias, accuracy = evaluate_metrics_with_prior(model, val_loader, device, smoothed_prior, alpha)

    maelist.append(mae)
    rmselist.append(rmse)
    biaslist.append(bias)
    accuracylist.append(accuracy)
    results.to_csv(f"1secondsalpha{alpha}.csv")
    results["error"] = results["predicted_magnitude"] - results["true_magnitude"]
    results["squared_error"] = results["error"] ** 2

    results["bin"] = pd.cut(
        results["true_magnitude"],
        bins=[0, 1, 2, 3, 4, 5, 6],
        labels=["0-1", "1-2", "2-3", "3-4", "4-5", "5-6"],
        include_lowest=True
    )

    bin_results = results.groupby("bin", observed=True).agg(
        bias=("error", "mean"),
        mse=("squared_error", "mean")
    )

    bin_results["rmse"] = np.sqrt(bin_results["mse"])

    row = {"alpha": alpha}

    for bin_name in ["0-1", "1-2", "2-3", "3-4", "4-5", "5-6"]:
        row[f"{bin_name} Bias"] = bin_results.loc[bin_name, "bias"] if bin_name in bin_results.index else np.nan
        row[f"{bin_name} RMSE"] = bin_results.loc[bin_name, "rmse"] if bin_name in bin_results.index else np.nan

    bin_rows.append(row)

bin_df = pd.DataFrame(bin_rows)

columns = ["alpha"] + [f"{b} Bias" for b in ["0-1", "1-2", "2-3", "3-4", "4-5", "5-6"]] + [f"{b} RMSE" for b in ["0-1", "1-2", "2-3", "3-4", "4-5", "5-6"]]
bin_df = bin_df[columns]

overall_df = pd.DataFrame({
    "alpha": alphas,
    "MAE": maelist,
    "RMSE": rmselist,
    "Bias": biaslist,
    "Accuracy ±0.2": accuracylist
})

display(overall_df)
display(bin_df)



,alpha,MAE,RMSE,Bias,Accuracy ±0.2
0,0.00,0.593288,0.762875,0.165536,22.170672
1,0.25,0.513793,0.661940,0.146359,25.266773
2,0.50,0.468666,0.605860,0.128986,28.132957
3,0.75,0.447327,0.582519,0.119140,30.624890
4,1.00,0.442075,0.580356,0.115163,32.517812
5,1.25,0.446999,0.591020,0.113935,33.613765


,alpha,0-1 Bias,1-2 Bias,2-3 Bias,3-4 Bias,4-5 Bias,5-6 Bias,0-1 RMSE,1-2 RMSE,2-3 RMSE,3-4 RMSE,4-5 RMSE,5-6 RMSE
0,0.00,0.676119,0.250190,0.078783,-0.065724,-0.289427,-0.602276,0.849865,0.660876,0.772860,0.933869,1.116747,1.419959
1,0.25,0.832802,0.311967,0.012576,-0.262446,-0.502472,-0.807892,0.933005,0.582856,0.613973,0.855011,1.171677,1.544350
2,0.50,0.947376,0.359740,-0.038237,-0.438082,-0.728515,-1.028924,1.010957,0.548263,0.497342,0.821816,1.240947,1.658923
3,0.75,1.039322,0.402594,-0.071925,-0.586580,-0.970697,-1.288100,1.083228,0.542895,0.414532,0.822288,1.324521,1.768365
4,1.00,1.117032,0.441322,-0.093358,-0.707699,-1.229947,-1.628318,1.149157,0.553348,0.357122,0.847553,1.432728,1.898579
5,1.25,1.182403,0.474218,-0.107846,-0.802511,-1.487040,-2.046118,1.207001,0.570097,0.320176,0.888106,1.579754,2.127410
